Lesson 3: Prompting for structure and setting up a retry method
In this notebook, you'll learn how to combine Pydantic with retry strategies to reliably extract structured output from an LLM.

By the end, you'll be able to:

Define structured data models for LLM responses
Build robust retry mechanisms for validation errors
Create reusable functions for LLM interactions

In [1]:
from google import genai

In [6]:
client = genai.Client()
response = client.models.generate_content(
  model="gemini-2.5-flash",contents="Explain how AI works in few words"
)
print(response.text)

AI learns from data to recognize patterns and make intelligent decisions or predictions.


In [7]:
from pydantic import BaseModel,ValidationError,EmailStr,Field
from typing import Literal,List,Optional
import json
from datetime import date
from dotenv import load_dotenv


In [20]:
#Define a JSON string representing the user input
user_input_json = '''
{
  "name":"Joe Doe",
  "email":"joe.doe@gmail.com",
  "query":"I forgot my password",
  "order_id":null,
  "purchase_date":null
}
'''
type(user_input_json)

str

In [21]:
#Define data model
class UserInput(BaseModel):
  name:str
  email:EmailStr
  query:str
  order_id:Optional[str]=Field(
    None,
    description="5 digit order number",
    ge=10000,
    le=99999
  )
  purchase_date:Optional[date]=None

In [22]:
user_input = UserInput.model_validate_json(user_input_json)
print(user_input)

name='Joe Doe' email='joe.doe@gmail.com' query='I forgot my password' order_id=None purchase_date=None


In [23]:
class CustomerQuery(UserInput):
  priority:str = Field(
    ...,description="Priority level: low,medium,high"
  )
  category:Literal[
    'refund_request','information_request','other'
  ] = Field(
    ...,description="Query category"
  )
  is_complaint:bool = Field(
    ...,description="Whether this is a complaint"
  )
  tags:List[str] = Field(
    ...,description="Relevant keyword tags"
  )

In [27]:
# Create a prompt with generic example data to guide LLM.
example_response_structure = f"""{{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}}"""

In [28]:
print(example_response_structure)

{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}


In [29]:
# Create prompt with user data and expected JSON structure
prompt = f"""
Please analyze this user query\n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{example_response_structure}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.
"""

print(prompt)


Please analyze this user query
 {
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.



In [30]:
def call_llm(prompt,model="gemini-3-flash-preview"):
  response = client.models.generate_content(model=model,contents=prompt)
  return response.text

In [32]:
response_content = call_llm(prompt)
print(response_content)

{
    "name": "Joe Doe",
    "email": "joe.doe@gmail.com",
    "query": "I forgot my password",
    "order_id": null,
    "purchase_date": null,
    "priority": "medium",
    "category": "account_access",
    "is_complaint": false,
    "tags": ["password", "account", "login"]
}


In [36]:
# Attempt to parse the response into CustomerQuery model
valid_data = CustomerQuery.model_validate_json(response_content)

ValidationError: 1 validation error for CustomerQuery
category
  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='account_access', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error

In [37]:
# Define a function to validate an LLM response
def validate_with_model(data_model,llm_response):
  try:
      validated_data = data_model.model_validate_json(llm_response)
      print("data validation successfull!")
      print(validated_data.model_dump_json(indent=2))
      return validated_data,None
  except ValidationError as e:
      print(f"error validating data: {e}")
      error_message = (
         f"This response generated a validation error:{e}"
      )
      return None,error_message

In [38]:
# Test your validation function with the LLM response
validated_data,validation_error = validate_with_model(CustomerQuery,response_content)

error validating data: 1 validation error for CustomerQuery
category
  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='account_access', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error


In [39]:
# Define a function to create a retry prompt with error feedback
def create_retry_prompt(original_prompt,original_response,error_message):
  retry_prompt = f"""
    This is a request to fix an error in the structure of an llm_response.
    Here is the original request:
    <original_prompt>
    {original_prompt}
    </original_prompt>

    Here is the original llm_response:
    <llm_response>
    {original_response}
    </llm_response>

    This response generated an error: 
    <error_message>
    {error_message}
    </error_message>

    Compare the error message and the llm_response and identify what 
    needs to be fixed or removed
    in the llm_response to resolve this error. 

    Respond ONLY with valid JSON. Do not include any explanations or 
    other text or formatting before or after the JSON string.
"""
  return retry_prompt

In [42]:
#Create a retry prompt for validation errors
validation_retry_prompt = create_retry_prompt(
  original_prompt=prompt,
  original_response=response_content,
  error_message=validation_error
)
print(validation_retry_prompt)


    This is a request to fix an error in the structure of an llm_response.
    Here is the original request:
    <original_prompt>
    
Please analyze this user query
 {
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.

    </original_prompt>

    Here is the original llm_response:
    <llm_response>
    {
    "name": "Joe Doe",
    "email": "j

In [43]:
# Call the LLM with the validation retry prompt
validation_retry_response = call_llm(validation_retry_prompt)
print(validation_retry_response)

{
    "name": "Joe Doe",
    "email": "joe.doe@gmail.com",
    "query": "I forgot my password",
    "order_id": null,
    "purchase_date": null,
    "priority": "medium",
    "category": "information_request",
    "is_complaint": false,
    "tags": ["password", "account", "login"]
}


In [45]:
#Attempt to validate the response from LLM after first retry
validated_response,validation_error = validate_with_model(
  CustomerQuery,validation_retry_response
)

data validation successfull!
{
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null,
  "priority": "medium",
  "category": "information_request",
  "is_complaint": false,
  "tags": [
    "password",
    "account",
    "login"
  ]
}


In [47]:
# Create a second retry prompt for validation errors
second_validation_retry_prompt = create_retry_prompt(
  original_prompt=validation_retry_prompt,
  original_response=validation_retry_response,
  error_message=validation_error
)
print(second_validation_retry_prompt)


    This is a request to fix an error in the structure of an llm_response.
    Here is the original request:
    <original_prompt>
    
    This is a request to fix an error in the structure of an llm_response.
    Here is the original request:
    <original_prompt>
    
Please analyze this user query
 {
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON

In [48]:
# Call the LLM with the second validation retry prompt
second_validation_retry_response = call_llm(
  second_validation_retry_prompt
)
print(second_validation_retry_response)

{
    "name": "Joe Doe",
    "email": "joe.doe@gmail.com",
    "query": "I forgot my password",
    "order_id": null,
    "purchase_date": null,
    "priority": "medium",
    "category": "information_request",
    "is_complaint": false,
    "tags": ["password", "account", "login"]
}


Define a function to handle multiple retries in a feedback loop

In [ ]:
# Define a function to automatically retry an LLM call multiple times
def validate_llm_response(
    prompt,data_model,n_retry=5,model="gemini-3-flash-preview"
):
  #Initial LLM call
  response_content = call_llm(prompt,model)
  current_prompt=prompt
  #Try to validate with the model
  #attempt: 0=initial,1=first retry
  for attempt in range(n_retry+1):
    validated_data,validation_error = validate_with_model(
      data_model,response_content
    )
    if validation_error:
      if attempt < n_retry:
        print(f"retry {attempt} of {n_retry} failed, trying again...")
      else:
        print(f"Max retries reached. Last error {validation_error}")
        return None,(
          f"Max retries reached. Last error {validation_error}"
        )
      validation_retry_prompt = create_retry_prompt(
        original_prompt=current_prompt,
        original_response=response_content,
        error_message=validation_error
      )

      response_content = call_llm(
        validation_retry_prompt,model=model
      )
      current_prompt=validation_retry_prompt
      continue

    return validated_data,None

In [51]:
# Test your complete solution with the original prompt
validated_data, error = validate_llm_response(
    prompt, CustomerQuery
)

error validating data: 1 validation error for CustomerQuery
category
  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='account_access', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error
retry 0 of 5 failed, trying again...
data validation successfull!
{
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null,
  "priority": "medium",
  "category": "information_request",
  "is_complaint": false,
  "tags": [
    "password",
    "login",
    "account"
  ]
}


In [52]:
# Investigate the model_json_schema for CustomerQuery
data_model_schema = json.dumps(
  CustomerQuery.model_json_schema(),indent=2
  )
print(data_model_schema)

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "email": {
      "format": "email",
      "title": "Email",
      "type": "string"
    },
    "query": {
      "title": "Query",
      "type": "string"
    },
    "order_id": {
      "anyOf": [
        {
          "ge": 10000,
          "le": 99999,
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "5 digit order number",
      "title": "Order Id"
    },
    "purchase_date": {
      "anyOf": [
        {
          "format": "date",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Purchase Date"
    },
    "priority": {
      "description": "Priority level: low,medium,high",
      "title": "Priority",
      "type": "string"
    },
    "category": {
      "description": "Query category",
      "enum": [
        "refund_request",
     

In [53]:
#Print the original prompt above
print(prompt)


Please analyze this user query
 {
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.



In [54]:
# Create new prompt with user input and model_json_schema
prompt = f"""
Please analyze this user query\n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching the following schema:
{data_model_schema}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.
"""

In [56]:
# Run your validate_llm_response function with the new prompt
final_analysis,error = validate_llm_response(
  prompt,CustomerQuery
)

data validation successfull!
{
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null,
  "priority": "medium",
  "category": "other",
  "is_complaint": false,
  "tags": [
    "password_reset",
    "account_access",
    "login_issue"
  ]
}
